# Week 1 — Python Foundations

You already know how to code (OOP, C++), so we **skim** the basics (variables, types, functions, control flow) and **go deep** where Pandas will lean on you every day:

- list comprehensions
- lambdas and functional patterns
- choosing the right container: dict vs list vs set

The Kaggle **Python** course is the companion track — take it this week and claim the certificate.
This notebook is the hands-on part, using `sample_data.csv` (a small orders table).

## Exercise 1 — Load a CSV by hand (no pandas yet)

Pandas is automation on top of what you're about to write by hand. Build the mental model first.

**Task:** write `load_orders(path)` that reads `sample_data.csv` and returns a **list of dicts** — one dict per row — with typed values:

- `order_id` → `int`
- `quantity` → `int`
- `price` → `float`
- `date` → keep as `str` (dates are a Pandas lesson, not today)

Then, on top of `orders`, produce:

1. `electronics_orders` — category is `electronics` **and** quantity ≥ 2
2. `revenue_by_category` — total `quantity * price` per category
3. print the category with the most revenue

**Hints:** `f.readline()` for the header, `line.split(",")`, `dict(zip(header, values))`, `int(...)` / `float(...)` casts, `dict.get(key, 0)` for accumulating.

Write it in the cell below **before** peeking at the solution.

In [ ]:
from pathlib import Path

DATA = Path("sample_data.csv")

# TODO 1 — load the CSV into a list of dicts, one per row.
#   read the header line, then for each row: dict(zip(header, values))
#   cast order_id -> int, quantity -> int, price -> float
def load_orders(path):
    orders = []
    # your code here
    return orders


orders = load_orders(DATA)
print(f"Loaded {len(orders)} orders")

# TODO 2 — electronics orders with quantity >= 2 (a list comprehension)
electronics_orders = []

# TODO 3 — total revenue per category (loop + dict.get, or a comprehension)
revenue_by_category = {}

# TODO 4 — print the category with the most revenue

### Solution — try the exercise first, then compare

In [ ]:
from pathlib import Path

DATA = Path("sample_data.csv")

def load_orders(path):
    """Read a CSV into a list of dicts, casting numeric fields."""
    orders = []
    with open(path, "r", encoding="utf-8") as f:
        header = f.readline().strip().split(",")
        for line in f:
            line = line.strip()
            if not line:            # skip blank lines
                continue
            row = dict(zip(header, line.split(",")))
            row["order_id"] = int(row["order_id"])
            row["quantity"] = int(row["quantity"])
            row["price"] = float(row["price"])
            orders.append(row)
    return orders


orders = load_orders(DATA)
print(f"Loaded {len(orders)} orders")

# 1. filter
electronics_orders = [
    o for o in orders
    if o["category"] == "electronics" and o["quantity"] >= 2
]
print(f"Electronics with qty >= 2: {len(electronics_orders)} order(s)")

# 2. revenue per category — dict.get(key, 0) is the classic accumulator
revenue_by_category = {}
for o in orders:
    revenue_by_category[o["category"]] = (
        revenue_by_category.get(o["category"], 0) + o["quantity"] * o["price"]
    )
print({k: round(v, 2) for k, v in revenue_by_category.items()})

# 3. biggest category
best = max(revenue_by_category, key=revenue_by_category.get)
print(f"Biggest category: {best} (${revenue_by_category[best]:.2f})")

## Deep Dive 1 — List comprehensions

Comprehensions collapse a 3-line loop into one expression. You'll see them in every Pandas/ML notebook:

- `[expr for x in iterable if cond]` — list
- `{expr for x in iterable}` — set (dedupes automatically)
- `{k: v for x in iterable}` — dict

Drills (run the Exercise 1 solution first so `orders` exists). Fill in the `...`:

1. squares of even numbers from 0 to 20
2. unique customer names containing the letter `"a"`
3. dict: category → number of orders
4. flatten `[[1, 2], [3, 4]]` into `[1, 2, 3, 4]`

In [ ]:
# 1. squares of even numbers from 0 to 20
even_squares = ...
print(even_squares)

# 2. unique customers whose name contains "a"
customers_with_a = ...
print(customers_with_a)

# 3. dict: category -> number of orders
order_counts = ...
print(order_counts)

# 4. flatten [[1, 2], [3, 4]] -> [1, 2, 3, 4]
flat = ...
print(flat)

### Comprehension solutions

In [ ]:
even_squares = [x * x for x in range(21) if x % 2 == 0]

customers_with_a = {o["customer"] for o in orders if "a" in o["customer"]}

order_counts = {
    c: sum(1 for o in orders if o["category"] == c)
    for c in {o["category"] for o in orders}
}

flat = [n for pair in [[1, 2], [3, 4]] for n in pair]

## Deep Dive 2 — Lambdas & functional patterns

A lambda is a tiny anonymous function: `lambda x: expr`. You'll meet it constantly in Pandas:

- `df.apply(lambda row: ...)`
- `df[col].map(lambda v: ...)`
- `sorted(data, key=lambda x: ...)`

Drills — **#1 is a worked example**, fill in the `...` for #2 and #3:

1. top 3 most expensive orders (worked)
2. the customer with the most total spend — build `spent` with the accumulator pattern, then `max(..., key=...)`
3. format prices as `"$12.75"` using `map(lambda ...)`

In [ ]:
# 1. worked example: top 3 most expensive orders
top3 = sorted(orders, key=lambda o: o["price"], reverse=True)[:3]
print([(o["order_id"], o["price"]) for o in top3])

# 2. biggest spender
spent = {}
for o in orders:
    spent[o["customer"]] = spent.get(o["customer"], 0) + o["quantity"] * o["price"]
biggest_spender = ...
print(biggest_spender)

# 3. prices formatted as "$12.75", first 5 orders
formatted = ...
print(formatted)

### Lambda solutions

In [ ]:
biggest_spender = max(spent, key=spent.get)
print(f"{biggest_spender} spent ${spent[biggest_spender]:.2f}")

formatted = list(map(lambda o: f"${o['price']:.2f}", orders[:5]))
print(formatted)

## dict vs list vs set — quick mental model

| Container | Use it when | Key operation |
|-----------|-------------|---------------|
| `list` | order matters, duplicates OK, index by position | `data[i]` |
| `dict` | look things up by a key (id, name, category) | `data[key]` |
| `set` | you only need membership or uniqueness | `x in data` |

Rule of thumb: if your only question is **"does this exist?"**, use a set. And `dict` is the workhorse of data work — much of Pandas is dicts under the hood.

In [ ]:
customers_list = [o["customer"] for o in orders]
customers_set = set(customers_list)

print(len(customers_list), "order rows vs", len(customers_set), "unique customers")
print("mia in customers_set:", "mia" in customers_set)

## Checkpoint — before moving on

- [ ] Python certificate claimed on Kaggle Learn
- [ ] Can write list comprehensions and lambdas without looking them up
- [ ] Can say when to use dict vs list vs set

**Recipe book note:** this manual loader is the hand-rolled version of `pd.read_csv()`. In the Pandas module we'll turn it into the reusable `patterns/01_load_explore.py` template.

**Next:** Module 2 — SQL + BigQuery. Tell the assistant when you're ready.